# Objective 3 — Step 5: Frozen Feature-Selection Replication

Step 4 selected the **group-aware Chi-Square Top-75% configuration on German Credit only**.

This notebook freezes that decision before examining feature-selection performance on:

- Australian Credit Approval
- Taiwan Credit Card Default

## Frozen configuration
- Selector: **Group-aware Chi-Square**
- Retained source-feature fraction: **75%**
- Evaluation classifier: **same fixed XGBoost configuration used in Step 3 and Step 4**
- No SMOTE
- No class weighting
- No hyperparameter tuning
- No probability calibration
- No threshold optimisation

The selector is retrained inside each outer-training fold because the datasets contain different features, but the **selector type, source-level aggregation rule, subset proportion, XGBoost configuration, and validation protocol are not retuned**.

This stage tests whether the German-developed feature-selection rule transfers to independent credit benchmarks.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
from itertools import combinations
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP4_DIR = BASE_DIR / "results" / "feature_selection_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "feature_selection_frozen_replication"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_RESULTS_FILE = BASELINE_DIR / "baseline_fold_results_all.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP4_DECISION_FILE = STEP4_DIR / "german_feature_selection_decision_table.csv"

DATASET_FILES = {
    "Australian Credit Approval":
        DATA_DIR / "australian_credit_approval_cleaned.csv",
    "Taiwan Credit Card Default":
        DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

required = [
    DICTIONARY_FILE,
    BASELINE_RESULTS_FILE,
    BASELINE_PREDICTIONS_FILE,
    STEP4_DECISION_FILE,
    *DATASET_FILES.values(),
]

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Required previous-step files are missing:\n"
        + "\n".join(missing)
    )

RANDOM_STATE = 42
REPEAT_SEEDS = [42, 142, 242, 342, 442]
N_FOLDS = 5

FROZEN_SELECTOR = "GroupAware_Chi2"
FROZEN_RETAINED_FRACTION = 0.75

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\feature_selection_frozen_replication


## 1. Lock and document the German-developed decision

In [3]:

step4_decision = pd.read_csv(STEP4_DECISION_FILE)

chosen_german = step4_decision[
    (step4_decision["selector"] == "Chi2")
    & (step4_decision["subset"] == "Top75")
].copy()

if len(chosen_german) != 1:
    raise RuntimeError(
        "Could not uniquely identify German Chi2 Top75 configuration."
    )

frozen_configuration = {
    "development_dataset": "German Credit",
    "selector": "Group-aware Chi-Square",
    "retained_source_feature_fraction": 0.75,
    "source_score_aggregation": (
        "Mean normalized transformed-variable relevance "
        "within each original source feature"
    ),
    "evaluation_classifier": (
        "Fixed XGBoost: n_estimators=300, max_depth=4, "
        "learning_rate=0.05, subsample=0.9, "
        "colsample_bytree=0.9"
    ),
    "external_replication_datasets": [
        "Australian Credit Approval",
        "Taiwan Credit Card Default",
    ],
    "no_external_retuning": True,
    "imbalance_treatment": "none",
    "calibration": "none",
    "threshold_optimization": "none",
    "german_development_evidence": {
        "selected_source_features": int(
            chosen_german["selected_source_features"].iloc[0]
        ),
        "feature_reduction_pct": float(
            chosen_german["feature_reduction_pct"].iloc[0]
        ),
        "roc_auc_mean": float(
            chosen_german["ROC_AUC_mean"].iloc[0]
        ),
        "pr_auc_mean": float(
            chosen_german["PR_AUC_mean"].iloc[0]
        ),
        "f1_mean": float(
            chosen_german["F1_mean"].iloc[0]
        ),
        "balanced_accuracy_mean": float(
            chosen_german["Balanced_Accuracy_mean"].iloc[0]
        ),
        "mcc_mean": float(
            chosen_german["MCC_mean"].iloc[0]
        ),
        "mean_jaccard": float(
            chosen_german["mean_jaccard"].iloc[0]
        ),
    },
}

with open(
    OUT_DIR / "frozen_feature_selection_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(frozen_configuration, file, indent=4)

print(json.dumps(frozen_configuration, indent=2))


{
  "development_dataset": "German Credit",
  "selector": "Group-aware Chi-Square",
  "retained_source_feature_fraction": 0.75,
  "source_score_aggregation": "Mean normalized transformed-variable relevance within each original source feature",
  "evaluation_classifier": "Fixed XGBoost: n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9",
  "external_replication_datasets": [
    "Australian Credit Approval",
    "Taiwan Credit Card Default"
  ],
  "no_external_retuning": true,
  "imbalance_treatment": "none",
  "calibration": "none",
  "threshold_optimization": "none",
  "german_development_evidence": {
    "selected_source_features": 15,
    "feature_reduction_pct": 25.0,
    "roc_auc_mean": 0.7884336394167986,
    "pr_auc_mean": 0.6222831394108276,
    "f1_mean": 0.5475793502837392,
    "balanced_accuracy_mean": 0.6839532576528906,
    "mcc_mean": 0.3987271330932572,
    "mean_jaccard": 0.8866666666666667
  }
}


## 2. Load cleaned replication datasets and feature roles

In [4]:

dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)

datasets = {
    name: pd.read_csv(path)
    for name, path in DATASET_FILES.items()
}

def roles_for(dataset_name):
    d = dictionary[dictionary["dataset"] == dataset_name].copy()

    return {
        role: (
            d.loc[d["role"] == role, "variable"]
            .astype(str)
            .tolist()
        )
        for role in [
            "categorical",
            "ordinal",
            "numerical",
            "identifier",
        ]
    }

roles = {
    name: roles_for(name)
    for name in datasets
}

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    print(
        dataset_name,
        "| records =", len(df),
        "| source predictors =", len(predictors),
        "| retained at 75% =",
        math.ceil(len(predictors) * FROZEN_RETAINED_FRACTION),
        "| adverse rate =",
        round(df["adverse_target"].mean(), 4),
    )


Australian Credit Approval | records = 690 | source predictors = 14 | retained at 75% = 11 | adverse rate = 0.5551
Taiwan Credit Card Default | records = 30000 | source predictors = 23 | retained at 75% = 18 | adverse rate = 0.2212


## 3. Preprocessing functions

In [5]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(
    dataset_name,
    mode,
    selected_features=None,
):
    r = roles[dataset_name]

    all_predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    if selected_features is None:
        selected_features = all_predictors

    selected_features = list(selected_features)

    selected_cat = [
        f for f in r["categorical"]
        if f in selected_features
    ]
    selected_ord = [
        f for f in r["ordinal"]
        if f in selected_features
    ]
    selected_num = [
        f for f in r["numerical"]
        if f in selected_features
    ]

    transformers = []

    if selected_num:
        if mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("num", num_pipe, selected_num)
        )

    if selected_ord:
        if mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("ord", ord_pipe, selected_ord)
        )

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])

        transformers.append(
            ("cat", cat_pipe, selected_cat)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


## 4. Dynamic transformed-to-source feature mapping

In [6]:

def feature_map_from_fitted_preprocessor(
    fitted_preprocessor,
    dataset_name,
    selected_features=None,
):
    r = roles[dataset_name]

    all_predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    if selected_features is None:
        selected_features = all_predictors

    selected_features = list(selected_features)

    selected_num = [
        f for f in r["numerical"]
        if f in selected_features
    ]
    selected_ord = [
        f for f in r["ordinal"]
        if f in selected_features
    ]
    selected_cat = [
        f for f in r["categorical"]
        if f in selected_features
    ]

    rows = []
    index = 0

    for feature in selected_num:
        rows.append({
            "transformed_index": index,
            "transformed_feature": f"num__{feature}",
            "source_feature": feature,
            "source_role": "numerical",
        })
        index += 1

    for feature in selected_ord:
        rows.append({
            "transformed_index": index,
            "transformed_feature": f"ord__{feature}",
            "source_feature": feature,
            "source_role": "ordinal",
        })
        index += 1

    if selected_cat:
        cat_pipeline = fitted_preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]
        encoded_names = encoder.get_feature_names_out(selected_cat)

        pos = 0
        for source_feature, categories in zip(
            selected_cat,
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": index,
                    "transformed_feature": "cat__" + str(encoded_names[pos]),
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })
                pos += 1
                index += 1

    result = pd.DataFrame(rows)

    assert len(result) == len(
        fitted_preprocessor.get_feature_names_out()
    )

    return result


## 5. Frozen group-aware Chi-Square ranking

In [7]:

def group_aware_chi2_ranking(
    dataset_name,
    X_train,
    y_train,
):
    prep = build_preprocessor(
        dataset_name,
        "chi2",
    )

    X_chi = prep.fit_transform(
        X_train,
        y_train,
    )

    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_preprocessor(
        prep,
        dataset_name,
    )

    raw_scores, _ = chi2(
        X_chi,
        y_train,
    )

    temp = fmap.copy()
    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)

    transformed_rank = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    if n > 1:
        temp["normalized_relevance"] = (
            1.0
            - (transformed_rank - 1.0) / (n - 1.0)
        )
    else:
        temp["normalized_relevance"] = 1.0

    grouped = (
        temp.groupby("source_feature", as_index=False)
        .agg(
            group_score=("normalized_relevance", "mean"),
            transformed_columns=("transformed_feature", "count"),
        )
    )

    grouped["source_rank"] = grouped["group_score"].rank(
        ascending=False,
        method="average",
    )

    grouped = grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)

    return grouped


## 6. Fixed evaluation classifier and metrics

In [8]:

def fresh_xgb():
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )


def calculate_metrics(
    y_true,
    y_pred,
    y_score,
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(sensitivity * specificity)
        if not np.isnan(sensitivity + specificity)
        else np.nan
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        y_score,
    )

    ks = float(np.max(tpr - fpr))

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_adverse": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall_adverse": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "mcc": matthews_corrcoef(
            y_true, y_pred
        ),
        "roc_auc": roc_auc_score(
            y_true, y_score
        ),
        "pr_auc": average_precision_score(
            y_true, y_score
        ),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 7. Recreate and verify the exact Step-3 folds

In [9]:

def build_and_verify_splits(
    dataset_name,
    df,
):
    r = roles[dataset_name]

    predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    X = df[predictors].copy()
    y = df["adverse_target"].astype(int).copy()
    groups = df["profile_group_id"].astype(str).copy()

    saved_predictions = baseline_predictions[
        (baseline_predictions["dataset"] == dataset_name)
        & (baseline_predictions["model"] == "XGB")
    ].copy()

    split_dict = {}

    for repeat_no, seed in enumerate(
        REPEAT_SEEDS,
        start=1,
    ):
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (train_idx, test_idx) in enumerate(
            splitter.split(X, y, groups),
            start=1,
        ):
            run_id = f"R{repeat_no}_F{fold_no}"

            expected_test = set(
                np.asarray(test_idx, dtype=int).tolist()
            )

            saved_test = set(
                saved_predictions.loc[
                    saved_predictions["run_id"] == run_id,
                    "source_row_index",
                ]
                .astype(int)
                .tolist()
            )

            assert expected_test == saved_test, (
                f"{dataset_name} {run_id} differs from Step 3."
            )

            assert len(
                set(groups.iloc[train_idx]).intersection(
                    set(groups.iloc[test_idx])
                )
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(train_idx, dtype=int),
                "test_idx": np.asarray(test_idx, dtype=int),
            }

    return split_dict


all_splits = {
    name: build_and_verify_splits(name, df)
    for name, df in datasets.items()
}

print("All Australian and Taiwan folds exactly match Step 3.")


All Australian and Taiwan folds exactly match Step 3.


## 8. Run frozen replication

In [10]:

ranking_rows = []
selection_rows = []
performance_rows = []

for dataset_name, df in datasets.items():
    r = roles[dataset_name]

    predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    X = df[predictors].copy()
    y = df["adverse_target"].astype(int).copy()

    p = len(predictors)
    n_select = int(
        math.ceil(
            p * FROZEN_RETAINED_FRACTION
        )
    )

    print("\n" + "=" * 80)
    print(dataset_name)
    print(
        f"Retaining {n_select}/{p} source features "
        f"({100*n_select/p:.2f}%)"
    )
    print("=" * 80)

    for run_number, (
        run_id,
        info,
    ) in enumerate(
        all_splits[dataset_name].items(),
        start=1,
    ):
        train_idx = info["train_idx"]
        test_idx = info["test_idx"]

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        selector_start = time.perf_counter()

        ranking = group_aware_chi2_ranking(
            dataset_name,
            X_train,
            y_train,
        )

        selector_runtime = (
            time.perf_counter() - selector_start
        )

        selected_features = (
            ranking["source_feature"]
            .astype(str)
            .tolist()[:n_select]
        )

        ranking_copy = ranking.copy()
        ranking_copy["dataset"] = dataset_name
        ranking_copy["run_id"] = run_id
        ranking_copy["repeat"] = info["repeat"]
        ranking_copy["fold"] = info["fold"]

        ranking_rows.extend(
            ranking_copy.to_dict("records")
        )

        selection_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "selector": FROZEN_SELECTOR,
            "retained_fraction": FROZEN_RETAINED_FRACTION,
            "total_source_features": p,
            "selected_source_features": n_select,
            "feature_reduction_pct": (
                100.0 * (1.0 - n_select / p)
            ),
            "selected_features": ";".join(selected_features),
        })

        eval_preprocessor = build_preprocessor(
            dataset_name,
            "tree",
            selected_features=selected_features,
        )

        pipeline = Pipeline([
            ("preprocessor", eval_preprocessor),
            ("model", fresh_xgb()),
        ])

        start = time.perf_counter()

        pipeline.fit(
            X_train[selected_features],
            y_train,
        )

        y_pred = pipeline.predict(
            X_test[selected_features]
        )

        y_score = pipeline.predict_proba(
            X_test[selected_features]
        )[:, 1]

        runtime = time.perf_counter() - start

        metric_values = calculate_metrics(
            y_test,
            y_pred,
            y_score,
        )

        transformed_count = len(
            pipeline.named_steps[
                "preprocessor"
            ].get_feature_names_out()
        )

        performance_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "selector": FROZEN_SELECTOR,
            "retained_fraction": FROZEN_RETAINED_FRACTION,
            "total_source_features": p,
            "selected_source_features": n_select,
            "selected_transformed_features": transformed_count,
            "feature_reduction_pct": (
                100.0 * (1.0 - n_select / p)
            ),
            "selector_runtime_seconds": selector_runtime,
            "xgb_runtime_seconds": runtime,
            **metric_values,
        })

        print(
            f"{run_id} ({run_number}/25): "
            f"ROC={metric_values['roc_auc']:.4f}, "
            f"PR={metric_values['pr_auc']:.4f}, "
            f"MCC={metric_values['mcc']:.4f}"
        )


rankings_all = pd.DataFrame(ranking_rows)
selected_sets = pd.DataFrame(selection_rows)
fold_results = pd.DataFrame(performance_rows)

rankings_all.to_csv(
    OUT_DIR / "frozen_replication_source_rankings.csv",
    index=False,
)

selected_sets.to_csv(
    OUT_DIR / "frozen_replication_selected_sets.csv",
    index=False,
)

fold_results.to_csv(
    OUT_DIR / "frozen_replication_fold_results.csv",
    index=False,
)

print("\nFrozen replication completed.")



Australian Credit Approval
Retaining 11/14 source features (78.57%)
R1_F1 (1/25): ROC=0.9190, PR=0.9503, MCC=0.7121
R1_F2 (2/25): ROC=0.9316, PR=0.9452, MCC=0.6893
R1_F3 (3/25): ROC=0.9345, PR=0.9488, MCC=0.7467
R1_F4 (4/25): ROC=0.9117, PR=0.8619, MCC=0.7025
R1_F5 (5/25): ROC=0.9411, PR=0.9552, MCC=0.7847
R2_F1 (6/25): ROC=0.9033, PR=0.9030, MCC=0.6217
R2_F2 (7/25): ROC=0.8988, PR=0.9278, MCC=0.6913
R2_F3 (8/25): ROC=0.9306, PR=0.9198, MCC=0.7824
R2_F4 (9/25): ROC=0.9162, PR=0.9082, MCC=0.6787
R2_F5 (10/25): ROC=0.9801, PR=0.9835, MCC=0.8096
R3_F1 (11/25): ROC=0.9144, PR=0.9343, MCC=0.6226
R3_F2 (12/25): ROC=0.9168, PR=0.9137, MCC=0.7529
R3_F3 (13/25): ROC=0.9410, PR=0.9568, MCC=0.6921
R3_F4 (14/25): ROC=0.9343, PR=0.9358, MCC=0.8105
R3_F5 (15/25): ROC=0.9223, PR=0.9115, MCC=0.6937
R4_F1 (16/25): ROC=0.9331, PR=0.9432, MCC=0.6937
R4_F2 (17/25): ROC=0.9168, PR=0.9345, MCC=0.6380
R4_F3 (18/25): ROC=0.9210, PR=0.8970, MCC=0.7246
R4_F4 (19/25): ROC=0.9148, PR=0.9511, MCC=0.6614
R4_F5 (20

## 9. Selection stability and frequency

In [11]:

def parse_set(value):
    return set(str(value).split(";"))


stability_rows = []
frequency_rows = []

for dataset_name, group in selected_sets.groupby("dataset"):
    sets = [
        parse_set(value)
        for value in group["selected_features"]
    ]

    pair_scores = []

    for a, b in combinations(sets, 2):
        union = a | b
        pair_scores.append(
            len(a & b) / len(union)
            if union else 1.0
        )

    stability_rows.append({
        "dataset": dataset_name,
        "pairwise_comparisons": len(pair_scores),
        "mean_jaccard": np.mean(pair_scores),
        "std_jaccard": np.std(pair_scores, ddof=1),
        "min_jaccard": np.min(pair_scores),
        "max_jaccard": np.max(pair_scores),
    })

    r = roles[dataset_name]
    predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    for feature in predictors:
        count = sum(
            feature in selected_set
            for selected_set in sets
        )

        frequency_rows.append({
            "dataset": dataset_name,
            "source_feature": feature,
            "selected_runs": count,
            "total_runs": len(sets),
            "selection_frequency": count / len(sets),
        })


stability = pd.DataFrame(stability_rows)
frequency = pd.DataFrame(frequency_rows)

stability.to_csv(
    OUT_DIR / "frozen_replication_stability.csv",
    index=False,
)

frequency.to_csv(
    OUT_DIR / "frozen_replication_selection_frequency.csv",
    index=False,
)

display(stability)

for dataset_name in datasets:
    print("\n", dataset_name)
    display(
        frequency[
            frequency["dataset"] == dataset_name
        ].sort_values(
            "selection_frequency",
            ascending=False,
        )
    )


,dataset,pairwise_comparisons,mean_jaccard,std_jaccard,min_jaccard,max_jaccard
0,Australian Credit Approval,300,0.962222,0.069896,0.833333,1.0
1,Taiwan Credit Card Default,300,0.943158,0.052551,0.894737,1.0



 Australian Credit Approval


,dataset,source_feature,selected_runs,total_runs,selection_frequency
1,Australian Credit Approval,a4,25,25,1.00
2,Australian Credit Approval,a5,25,25,1.00
3,Australian Credit Approval,a6,25,25,1.00
4,Australian Credit Approval,a8,25,25,1.00
5,Australian Credit Approval,a9,25,25,1.00
8,Australian Credit Approval,a2,25,25,1.00
9,Australian Credit Approval,a3,25,25,1.00
10,Australian Credit Approval,a7,25,25,1.00
11,Australian Credit Approval,a10,25,25,1.00
13,Australian Credit Approval,a14,25,25,1.00



 Taiwan Credit Card Default


,dataset,source_feature,selected_runs,total_runs,selection_frequency
14,Taiwan Credit Card Default,sex,25,25,1.00
23,Taiwan Credit Card Default,limit_bal,25,25,1.00
35,Taiwan Credit Card Default,pay_amt5,25,25,1.00
34,Taiwan Credit Card Default,pay_amt4,25,25,1.00
33,Taiwan Credit Card Default,pay_amt3,25,25,1.00
32,Taiwan Credit Card Default,pay_amt2,25,25,1.00
31,Taiwan Credit Card Default,pay_amt1,25,25,1.00
15,Taiwan Credit Card Default,education,25,25,1.00
36,Taiwan Credit Card Default,pay_amt6,25,25,1.00
22,Taiwan Credit Card Default,pay_6,25,25,1.00


## 10. Paired comparison with Step-3 all-feature XGBoost

In [12]:

baseline_xgb = baseline_results[
    (baseline_results["model"] == "XGB")
    & (
        baseline_results["dataset"].isin(
            list(datasets.keys())
        )
    )
][
    [
        "dataset",
        "run_id",
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "precision_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
        "gmean",
        "ks_statistic",
    ]
].copy()

baseline_xgb = baseline_xgb.rename(
    columns={
        column: (
            "baseline_" + column
            if column not in [
                "dataset",
                "run_id",
            ]
            else column
        )
        for column in baseline_xgb.columns
    }
)

paired = fold_results.merge(
    baseline_xgb,
    on=[
        "dataset",
        "run_id",
    ],
    how="left",
    validate="one_to_one",
)

comparison_metrics = [
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "ks_statistic",
]

for metric in comparison_metrics:
    paired["delta_" + metric] = (
        paired[metric]
        - paired["baseline_" + metric]
    )

paired.to_csv(
    OUT_DIR / "frozen_replication_paired_deltas.csv",
    index=False,
)

summary = (
    paired.groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        Selected_Source_Features=("selected_source_features", "mean"),
        Feature_Reduction_Pct=("feature_reduction_pct", "mean"),
        ROC_AUC=("roc_auc", "mean"),
        Baseline_ROC_AUC=("baseline_roc_auc", "mean"),
        Delta_ROC_AUC=("delta_roc_auc", "mean"),
        PR_AUC=("pr_auc", "mean"),
        Baseline_PR_AUC=("baseline_pr_auc", "mean"),
        Delta_PR_AUC=("delta_pr_auc", "mean"),
        Recall=("recall_adverse", "mean"),
        Baseline_Recall=("baseline_recall_adverse", "mean"),
        Delta_Recall=("delta_recall_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Baseline_F1=("baseline_f1_adverse", "mean"),
        Delta_F1=("delta_f1_adverse", "mean"),
        Balanced_Accuracy=("balanced_accuracy", "mean"),
        Baseline_Balanced_Accuracy=("baseline_balanced_accuracy", "mean"),
        Delta_Balanced_Accuracy=("delta_balanced_accuracy", "mean"),
        MCC=("mcc", "mean"),
        Baseline_MCC=("baseline_mcc", "mean"),
        Delta_MCC=("delta_mcc", "mean"),
    )
)

summary = summary.merge(
    stability[
        [
            "dataset",
            "mean_jaccard",
        ]
    ],
    on="dataset",
    how="left",
)

summary.to_csv(
    OUT_DIR / "frozen_replication_summary.csv",
    index=False,
)

display(summary)


,dataset,Selected_Source_Features,Feature_Reduction_Pct,ROC_AUC,Baseline_ROC_AUC,Delta_ROC_AUC,PR_AUC,Baseline_PR_AUC,Delta_PR_AUC,Recall,...,F1,Baseline_F1,Delta_F1,Balanced_Accuracy,Baseline_Balanced_Accuracy,Delta_Balanced_Accuracy,MCC,Baseline_MCC,Delta_MCC,mean_jaccard
0,Australian Credit Approval,11.0,21.428571,0.925940,0.931568,-0.005628,0.930499,0.932837,-0.002338,0.871046,...,0.873366,0.883336,-0.009970,0.859510,0.872130,-0.012620,0.717740,0.742331,-0.024592,0.962222
1,Taiwan Credit Card Default,18.0,21.739130,0.782617,0.783234,-0.000617,0.560105,0.560525,-0.000420,0.366999,...,0.475385,0.475333,0.000051,0.658392,0.658417,-0.000025,0.404185,0.402911,0.001274,0.943158


## 11. Interpretation aid

Do not automatically declare success or failure from one metric.

The replication will be reviewed jointly for:

- discrimination: ROC-AUC and PR-AUC;
- adverse-class behavior: recall and F1;
- threshold-balanced performance: balanced accuracy and MCC;
- feature reduction;
- source-feature stability.

If performance is not preserved on one external dataset, the frozen design is **not retuned**. That result is scientifically important and will guide the later hybrid framework.


In [13]:

# Internal consistency checks
assert len(fold_results) == 2 * 25
assert len(selected_sets) == 2 * 25

assert fold_results["roc_auc"].between(0, 1).all()
assert fold_results["pr_auc"].between(0, 1).all()
assert fold_results["mcc"].between(-1, 1).all()

assert not paired[
    [
        "baseline_roc_auc",
        "baseline_pr_auc",
        "baseline_mcc",
    ]
].isna().any().any()

manifest = []

for path in sorted(OUT_DIR.iterdir()):
    if path.is_file():
        manifest.append(path.name)

pd.DataFrame(
    {"generated_file": manifest}
).to_csv(
    OUT_DIR / "step5_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 5 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("\nOutput folder:")
print(OUT_DIR)
print("\nMost important file:")
print(" - frozen_replication_summary.csv")
print(" - frozen_replication_stability.csv")
print(" - frozen_replication_paired_deltas.csv")
print(" - frozen_replication_selection_frequency.csv")


STEP 5 COMPLETED SUCCESSFULLY

Output folder:
D:\PHD\Research Paper writing\3rd Obj. paper\results\feature_selection_frozen_replication

Most important file:
 - frozen_replication_summary.csv
 - frozen_replication_stability.csv
 - frozen_replication_paired_deltas.csv
 - frozen_replication_selection_frequency.csv
